# Medical Healthcare Assistant - LLM Fine-Tuning with LoRA

**Project**: Domain-Specific Assistant via LLM Fine-Tuning

**Objective**: Fine-tune TinyLlama-1.1B on medical Q&A data using LoRA for efficient parameter-efficient training

**Key Features**:
- Domain-restricted responses (medical topics only)
- Text-based progress bars (compatible everywhere)
- LoRA fine-tuning with 4-bit quantization
- Comprehensive evaluation metrics

---

## Table of Contents

1. [Setup & Installation](#1-setup)
2. [Progress Bar Configuration](#2-config)
3. [Domain Classification](#3-domain)
4. [Dataset Loading](#4-dataset)
5. [Data Preprocessing](#5-preprocessing)
6. [Base Model Testing](#6-base)
7. [Model Fine-Tuning with LoRA](#7-training)
8. [Model Evaluation](#8-evaluation)
9. [Base vs Fine-Tuned Comparison](#9-comparison)
10. [Gradio UI Deployment](#10-deployment)
11. [Conclusion](#11-conclusion)

---
## 1. Setup & Installation

Install all required libraries for fine-tuning and evaluation.

In [ ]:
%%capture
# Install required packages
!pip install -q torch transformers datasets peft bitsandbytes accelerate
!pip install -q evaluate rouge-score sacrebleu nltk
!pip install -q gradio matplotlib seaborn plotly
!pip install -q scipy scikit-learn

In [ ]:
# Import all necessary libraries
import os
import json
import warnings
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# PyTorch and Transformers
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    pipeline
)

# PEFT for LoRA
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)

# Dataset and metrics
from datasets import load_dataset, Dataset
import evaluate

# UI
import gradio as gr

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {__import__('transformers').__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

---
## 2. Progress Bar Configuration

Configure text-based progress bars for compatibility with VS Code, GitHub, and Colab.

In [ ]:
# Configure for text-based progress bars (compatible everywhere)
import sys

# Force tqdm to use text mode instead of widgets
os.environ["TQDM_DISABLE"] = "0"  # Keep progress bars enabled
os.environ["TQDM_MININTERVAL"] = "1"  # Update every second

# Override notebook tqdm with standard tqdm
from tqdm import tqdm as std_tqdm
sys.modules['tqdm.notebook'] = sys.modules['tqdm']
sys.modules['tqdm.auto'] = sys.modules['tqdm']

print("Progress bar configuration: Text-based mode enabled")
print("This ensures compatibility with VS Code, GitHub, and Colab")

---
## 3. Domain Classification Function

Implement domain restriction to ensure the model only answers medical questions.

In [ ]:
# Domain Classification Function
def is_medical_question(question: str) -> tuple:
    """
    Check if a question is related to the medical/healthcare domain.
    
    Args:
        question: User input question
        
    Returns:
        tuple: (is_medical: bool, reason: str)
    """
    question_lower = question.lower().strip()
    
    # Medical keywords - comprehensive list
    medical_keywords = [
        # Body systems and anatomy
        'heart', 'lung', 'liver', 'kidney', 'brain', 'blood', 'bone', 'muscle', 'nerve',
        'stomach', 'intestine', 'pancreas', 'skin', 'eye', 'ear', 'throat', 'mouth',
        'artery', 'vein', 'organ', 'cell', 'tissue', 'gland', 'hormone', 'enzyme',
        'mitochondria', 'dna', 'gene', 'chromosome', 'protein', 'immune', 'lymph',
        
        # Medical terms and conditions
        'disease', 'disorder', 'syndrome', 'condition', 'infection', 'inflammation',
        'cancer', 'tumor', 'diabetes', 'hypertension', 'asthma', 'allergy', 'arthritis',
        'pneumonia', 'bronchitis', 'stroke', 'seizure', 'fracture', 'wound', 'injury',
        'pain', 'fever', 'cough', 'bleeding', 'swelling', 'rash', 'itch',
        
        # Medical procedures and treatments
        'treatment', 'therapy', 'surgery', 'operation', 'procedure', 'diagnosis',
        'medication', 'drug', 'medicine', 'antibiotic', 'vaccine', 'immunization',
        'prescription', 'dose', 'side effect', 'symptom', 'sign', 'test', 'exam',
        'screening', 'imaging', 'xray', 'x-ray', 'mri', 'ct scan', 'ultrasound',
        
        # Healthcare professionals and specialties
        'doctor', 'physician', 'nurse', 'surgeon', 'therapist', 'patient', 'hospital',
        'clinic', 'emergency', 'icu', 'pharmacy', 'medical', 'clinical', 'health',
        'healthcare', 'medicine', 'anatomy', 'physiology', 'pathology', 'pharmacology',
    ]
    
    # Non-medical topics to explicitly reject
    non_medical_topics = [
        'politics', 'political', 'election', 'government', 'president', 'minister',
        'religion', 'religious', 'god', 'allah', 'buddha', 'church', 'mosque', 'temple',
        'math', 'mathematics', 'algebra', 'calculus', 'geometry', 'equation', 'solve',
        'sports', 'football', 'basketball', 'soccer', 'cricket', 'game', 'team',
        'cooking', 'recipe', 'cuisine', 'restaurant',
        'programming', 'code', 'software', 'computer', 'python', 'java', 'javascript',
        'history', 'historical', 'war', 'battle', 'civilization',
        'geography', 'country', 'capital', 'continent', 'ocean',
        'entertainment', 'movie', 'film', 'music', 'song', 'actor', 'celebrity',
    ]
    
    # First check for non-medical topics
    for topic in non_medical_topics:
        if topic in question_lower:
            # Exception: nutrition/diet can be medical
            if 'diet' in question_lower or 'nutrition' in question_lower or 'vitamin' in question_lower:
                continue
            return False, f"This question appears to be about {topic}, which is outside my medical domain."
    
    # Check for medical keywords
    for keyword in medical_keywords:
        if keyword in question_lower:
            return True, "Medical question detected"
    
    # If no clear medical keywords but also no clear non-medical topics
    if len(question.split()) < 3:
        return False, "Please ask a complete medical question."
    
    # Default to rejecting if uncertain
    return False, "I can only answer questions related to medical and healthcare topics."

print("Domain classification function loaded")
print("The model will now restrict responses to medical/healthcare topics only")
print("\nTest examples:")
print("  Medical: 'What are the side effects of aspirin?' ->", is_medical_question("What are the side effects of aspirin?")[0])
print("  Non-medical: 'Who is the president?' ->", is_medical_question("Who is the president?")[0])

---
## 4. Dataset Loading & Exploration

Load the medical flashcards dataset from Hugging Face.

In [ ]:
# Load medical flashcards dataset
print("Loading medical flashcards dataset...")
dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards", split="train")

print(f"\nDataset loaded successfully!")
print(f"Total samples: {len(dataset):,}")
print(f"\nDataset features: {dataset.features}")
print(f"\nSample example:")
print(dataset[0])

In [ ]:
# Explore the dataset structure
print("Dataset Analysis:\n")

# Check for missing values
df = pd.DataFrame(dataset)
print(f"Missing values: {df.isnull().sum().sum()}")

# Analyze text lengths
df['input_length'] = df['input'].apply(lambda x: len(x.split()))
df['output_length'] = df['output'].apply(lambda x: len(x.split()))

print(f"\nInput Length Statistics:")
print(df['input_length'].describe())
print(f"\nOutput Length Statistics:")
print(df['output_length'].describe())

# Visualize length distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['input_length'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Input Length (words)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Input Lengths')
axes[0].axvline(df['input_length'].mean(), color='red', linestyle='--', label=f'Mean: {df["input_length"].mean():.0f}')
axes[0].legend()

axes[1].hist(df['output_length'], bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Output Length (words)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Output Lengths')
axes[1].axvline(df['output_length'].mean(), color='red', linestyle='--', label=f'Mean: {df["output_length"].mean():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Display sample conversations
print("Sample Medical Q&A Pairs:\n")
print("=" * 80)

for i in range(3):
    sample = dataset[i * 1000]  # Sample from different parts
    print(f"\nExample {i+1}:")
    print(f"\nQuestion: {sample['input'][:200]}..." if len(sample['input']) > 200 else f"\nQuestion: {sample['input']}")
    print(f"\nAnswer: {sample['output'][:300]}..." if len(sample['output']) > 300 else f"\nAnswer: {sample['output']}")
    print("\n" + "=" * 80)

---
## 5. Data Preprocessing

Prepare the dataset for training with proper formatting and tokenization.

In [ ]:
# Configuration
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_LENGTH = 512
TRAIN_SIZE = 5000
VAL_SIZE = 500
TEST_SIZE = 500

print(f"Model: {MODEL_NAME}")
print(f"Training samples: {TRAIN_SIZE:,}")
print(f"Validation samples: {VAL_SIZE:,}")
print(f"Test samples: {TEST_SIZE:,}")
print(f"Max sequence length: {MAX_LENGTH}")

In [ ]:
# Load tokenizer
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer loaded!")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"PAD token: {tokenizer.pad_token}")
print(f"EOS token: {tokenizer.eos_token}")

In [ ]:
# Create prompt template with enhanced system message
def create_prompt(instruction, response=""):
    """
    Create a formatted prompt for medical Q&A with domain boundaries.
    """
    # Enhanced system message with domain restrictions
    system_message = (
        "You are a specialized medical healthcare assistant. "
        "Your expertise is strictly limited to medical, healthcare, anatomy, physiology, "
        "diseases, treatments, medications, and related health topics. "
        "Provide accurate, detailed medical information. "
        "If asked about non-medical topics, politely decline and remind the user of your medical specialization."
    )
    
    if response:
        prompt = f"<|system|>\n{system_message}</s>\n<|user|>\n{instruction}</s>\n<|assistant|>\n{response}</s>"
    else:
        prompt = f"<|system|>\n{system_message}</s>\n<|user|>\n{instruction}</s>\n<|assistant|>\n"
    
    return prompt

print("Enhanced prompt template with domain boundaries created")

In [ ]:
# Preprocessing function
def preprocess_function(examples):
    """
    Tokenize and format the dataset.
    """
    # Create prompts
    prompts = []
    for inp, out in zip(examples['input'], examples['output']):
        prompt = create_prompt(inp, out)
        prompts.append(prompt)
    
    # Tokenize (without return_tensors for batched mapping)
    tokenized = tokenizer(
        prompts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    
    # Add labels (same as input_ids for causal LM)
    tokenized["labels"] = [ids.copy() for ids in tokenized["input_ids"]]
    
    return tokenized

print("Preprocessing function defined!")

In [ ]:
# Split dataset
print("Splitting dataset...\n")

# Shuffle and select samples
dataset_shuffled = dataset.shuffle(seed=SEED)

# Create splits
train_dataset = dataset_shuffled.select(range(TRAIN_SIZE))
val_dataset = dataset_shuffled.select(range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE))
test_dataset = dataset_shuffled.select(range(TRAIN_SIZE + VAL_SIZE, TRAIN_SIZE + VAL_SIZE + TEST_SIZE))

print(f"Train set: {len(train_dataset):,} samples")
print(f"Validation set: {len(val_dataset):,} samples")
print(f"Test set: {len(test_dataset):,} samples")

In [ ]:
# Tokenize datasets
print("\nTokenizing datasets...\n")

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing training data"
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation data"
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
    desc="Tokenizing test data"
)

print("\nTokenization complete!")
print(f"\nTokenized training sample length: {len(tokenized_train[0]['input_ids'])} tokens")

---
## 6. Base Model Testing

Test the base pre-trained model before fine-tuning to establish a baseline.

In [ ]:
# Load base model for testing
print("Loading base model for testing...\n")

base_model_test = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Base model loaded!")
print(f"Model parameters: {base_model_test.num_parameters():,}")

In [ ]:
# Test base model with sample medical questions
def generate_response(model, tokenizer, question, max_new_tokens=200, enforce_domain=True):
    """
    Generate a response from the model with domain checking.
    """
    # Check if question is in medical domain
    if enforce_domain:
        is_medical, reason = is_medical_question(question)
        if not is_medical:
            return (f"I apologize, but I can only answer questions related to medical and healthcare topics. "
                    f"{reason}\n\nPlease ask me about medical conditions, treatments, anatomy, "
                    f"medications, or other health-related topics.")
    
    # Generate response
    prompt = create_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the assistant's response
    if "<|assistant|>" in response:
        response = response.split("<|assistant|>")[-1].strip()
    
    return response

# Test questions
test_questions = [
    "What is the function of the mitochondria?",
    "What are the side effects of beta blockers?",
    "Explain what diabetes mellitus is."
]

print("Testing Base Model:\n")
print("=" * 80)

base_responses = []
for i, question in enumerate(test_questions, 1):
    print(f"\nQuestion {i}: {question}")
    response = generate_response(base_model_test, tokenizer, question, enforce_domain=False)
    base_responses.append(response)
    print(f"\nBase Model Response:\n{response}")
    print("\n" + "=" * 80)

# Clean up memory
del base_model_test
torch.cuda.empty_cache()
print("\nBase model removed from memory")

In [ ]:
# Test domain classification
print("Testing Domain Classification:\n")
print("=" * 80)

test_cases = [
    ("What are the side effects of aspirin?", "Medical"),
    ("Who is the president of the United States?", "Politics"),
    ("Solve 2x + 5 = 15", "Mathematics"),
    ("What causes diabetes?", "Medical"),
    ("Tell me about the movie Inception", "Entertainment")
]

for question, expected_domain in test_cases:
    is_medical, reason = is_medical_question(question)
    status = "ACCEPTED" if is_medical else "REJECTED"
    print(f"\n[{status}] {question}")
    print(f"Expected: {expected_domain} | Classification: {'Medical' if is_medical else 'Non-medical'}")
    if not is_medical:
        print(f"Reason: {reason}")
    print("-" * 80)

---
## 7. Model Fine-Tuning with LoRA

Fine-tune the model using LoRA (Low-Rank Adaptation) for parameter-efficient training.

In [ ]:
# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("4-bit Quantization Configuration:")
print("  - Quantization type: nf4")
print("  - Compute dtype: float16")
print("  - Double quantization: True")

In [ ]:
# Load model with quantization
print("\nLoading model with 4-bit quantization...\n")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

print("Model loaded and prepared!")
print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=16,                           # Rank
    lora_alpha=32,                  # Alpha (scaling)
    target_modules=[                # Target attention modules
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,              # Dropout
    bias="none",
    task_type="CAUSAL_LM"
)

print("LoRA Configuration:")
print(f"  - Rank (r): {lora_config.r}")
print(f"  - Alpha: {lora_config.lora_alpha}")
print(f"  - Target modules: {lora_config.target_modules}")
print(f"  - Dropout: {lora_config.lora_dropout}")

In [ ]:
# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("\nModel Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Trainable %: {100 * trainable_params / total_params:.4f}%")

model.print_trainable_parameters()

In [ ]:
# Training configuration
OUTPUT_DIR = "./medical-assistant-lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    fp16=True,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_steps=50,
    warmup_steps=100,
    weight_decay=0.01,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    save_total_limit=2,
    load_best_model_at_end=True,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=SEED
)

print("Training Configuration:")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Optimizer: {training_args.optim}")

In [ ]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

print("Trainer created successfully!")
print(f"\nTraining will process {len(tokenized_train)} samples")
print(f"Validation will use {len(tokenized_val)} samples")

# Calculate training time estimate
steps_per_epoch = len(tokenized_train) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
total_steps = steps_per_epoch * training_args.num_train_epochs
print(f"\nEstimated steps: {total_steps:,}")
print(f"Expected training time: ~2-3 hours on T4 GPU")

In [ ]:
# Train the model
print("\nStarting training...\n")
print("=" * 80)

train_result = trainer.train()

print("\n" + "=" * 80)
print("\nTraining completed!")
print(f"\nTraining Results:")
print(f"  - Final training loss: {train_result.training_loss:.4f}")
print(f"  - Training time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"  - Samples per second: {train_result.metrics['train_samples_per_second']:.2f}")

In [ ]:
# Save the fine-tuned model
print("\nSaving fine-tuned model...")

model.save_pretrained(OUTPUT_DIR + "/final_model")
tokenizer.save_pretrained(OUTPUT_DIR + "/final_model")

print(f"Model saved to {OUTPUT_DIR}/final_model")

In [ ]:
# Visualize training history
log_history = trainer.state.log_history

# Extract metrics
train_loss = [log['loss'] for log in log_history if 'loss' in log]
eval_loss = [log['eval_loss'] for log in log_history if 'eval_loss' in log]

# Plot training curves
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

if train_loss:
    ax.plot(train_loss, label='Training Loss', marker='o', linewidth=2)
if eval_loss:
    eval_steps = [i * (len(train_loss) // len(eval_loss)) for i in range(len(eval_loss))]
    ax.plot(eval_steps, eval_loss, label='Validation Loss', marker='s', linewidth=2)

ax.set_xlabel('Training Steps', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal validation loss: {eval_loss[-1]:.4f}" if eval_loss else "No validation loss recorded")

---
## 8. Model Evaluation

Evaluate the fine-tuned model using NLP metrics (BLEU, ROUGE, Perplexity).

In [ ]:
# Load evaluation metrics
import nltk
nltk.download('punkt', quiet=True)

bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

print("Evaluation metrics loaded!")

In [ ]:
# Generate predictions on test set
print("Generating predictions on test set...\n")

predictions = []
references = []

# Sample from test set for evaluation
eval_samples = 100

for i in tqdm(range(min(eval_samples, len(test_dataset))), desc="Generating predictions"):
    sample = test_dataset[i]
    
    # Generate prediction
    prompt = create_prompt(sample['input'])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract assistant response
    if "<|assistant|>" in pred_text:
        pred_text = pred_text.split("<|assistant|>")[-1].strip()
    
    predictions.append(pred_text)
    references.append(sample['output'])

print(f"\nGenerated {len(predictions)} predictions")

In [ ]:
# Calculate BLEU score
print("\nCalculating BLEU score...")

# Format references for BLEU
references_bleu = [[ref] for ref in references]

bleu_results = bleu_metric.compute(
    predictions=predictions,
    references=references_bleu
)

print(f"\nBLEU Score: {bleu_results['bleu']:.4f}")
print(f"  - BLEU-1: {bleu_results['precisions'][0]:.4f}")
print(f"  - BLEU-2: {bleu_results['precisions'][1]:.4f}")
print(f"  - BLEU-3: {bleu_results['precisions'][2]:.4f}")
print(f"  - BLEU-4: {bleu_results['precisions'][3]:.4f}")

In [ ]:
# Calculate ROUGE scores
print("\nCalculating ROUGE scores...")

rouge_results = rouge_metric.compute(
    predictions=predictions,
    references=references
)

print(f"\nROUGE Scores:")
print(f"  - ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"  - ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"  - ROUGE-L: {rouge_results['rougeL']:.4f}")

In [ ]:
# Calculate perplexity
print("\nCalculating perplexity...")

eval_results = trainer.evaluate()
perplexity = np.exp(eval_results['eval_loss'])

print(f"\nPerplexity: {perplexity:.4f}")
print(f"  - Evaluation Loss: {eval_results['eval_loss']:.4f}")

In [ ]:
# Create evaluation summary
evaluation_summary = {
    "BLEU": bleu_results['bleu'],
    "ROUGE-1": rouge_results['rouge1'],
    "ROUGE-2": rouge_results['rouge2'],
    "ROUGE-L": rouge_results['rougeL'],
    "Perplexity": perplexity,
    "Eval Loss": eval_results['eval_loss']
}

print("\n" + "=" * 80)
print("\nEVALUATION SUMMARY\n")
print("=" * 80)

for metric, value in evaluation_summary.items():
    print(f"{metric:20s}: {value:.4f}")

print("\n" + "=" * 80)

# Save evaluation results
with open(OUTPUT_DIR + "/evaluation_results.json", "w") as f:
    json.dump(evaluation_summary, f, indent=2)

print(f"\nEvaluation results saved to {OUTPUT_DIR}/evaluation_results.json")

In [ ]:
# Visualize evaluation metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BLEU components
bleu_scores = [bleu_results['bleu']] + bleu_results['precisions']
bleu_labels = ['Overall', 'BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4']

axes[0].bar(bleu_labels, bleu_scores, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('BLEU Scores', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, max(bleu_scores) * 1.2)
axes[0].grid(axis='y', alpha=0.3)

# ROUGE scores
rouge_scores = [rouge_results['rouge1'], rouge_results['rouge2'], rouge_results['rougeL']]
rouge_labels = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']

axes[1].bar(rouge_labels, rouge_scores, color='lightcoral', edgecolor='black', alpha=0.7)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('ROUGE Scores', fontsize=14, fontweight='bold')
axes[1].set_ylim(0, max(rouge_scores) * 1.2)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Base vs Fine-Tuned Comparison

Compare responses from the base model and fine-tuned model.

In [ ]:
# Reload base model for comparison
print("Loading base model for comparison...\n")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Base model loaded!")

In [ ]:
# Comparison questions
comparison_questions = [
    "What is the function of the mitochondria?",
    "What are the side effects of beta blockers?",
    "Explain what diabetes mellitus is.",
    "What is the treatment for hypertension?",
    "Describe the stages of wound healing."
]

print("Comparing Base Model vs Fine-Tuned Model\n")
print("=" * 100)

comparison_results = []

for i, question in enumerate(comparison_questions, 1):
    print(f"\nQuestion {i}: {question}\n")
    
    # Base model response
    base_response = generate_response(base_model, tokenizer, question, max_new_tokens=200, enforce_domain=False)
    print(f"Base Model:")
    print(f"{base_response}\n")
    
    # Fine-tuned model response
    finetuned_response = generate_response(model, tokenizer, question, max_new_tokens=200, enforce_domain=False)
    print(f"Fine-Tuned Model:")
    print(f"{finetuned_response}\n")
    
    comparison_results.append({
        "question": question,
        "base_response": base_response,
        "finetuned_response": finetuned_response
    })
    
    print("=" * 100)

# Save comparison results
with open(OUTPUT_DIR + "/comparison_results.json", "w") as f:
    json.dump(comparison_results, f, indent=2)

print(f"\nComparison results saved to {OUTPUT_DIR}/comparison_results.json")

In [ ]:
# Quantitative comparison on test samples
print("\nQuantitative Comparison on Test Set\n")

# Generate predictions with base model
base_predictions = []

print("Generating base model predictions...")
for i in tqdm(range(min(50, len(test_dataset))), desc="Base model"):
    sample = test_dataset[i]
    response = generate_response(base_model, tokenizer, sample['input'], max_new_tokens=150, enforce_domain=False)
    base_predictions.append(response)

# Calculate metrics for base model
base_bleu = bleu_metric.compute(
    predictions=base_predictions,
    references=[[ref] for ref in [test_dataset[i]['output'] for i in range(len(base_predictions))]]
)

base_rouge = rouge_metric.compute(
    predictions=base_predictions,
    references=[test_dataset[i]['output'] for i in range(len(base_predictions))]
)

print("\n" + "=" * 80)
print("\nCOMPARISON SUMMARY\n")
print("=" * 80)

comparison_table = pd.DataFrame({
    "Metric": ["BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L"],
    "Base Model": [
        base_bleu['bleu'],
        base_rouge['rouge1'],
        base_rouge['rouge2'],
        base_rouge['rougeL']
    ],
    "Fine-Tuned Model": [
        bleu_results['bleu'],
        rouge_results['rouge1'],
        rouge_results['rouge2'],
        rouge_results['rougeL']
    ]
})

comparison_table['Improvement (%)'] = (
    (comparison_table['Fine-Tuned Model'] - comparison_table['Base Model']) /
    comparison_table['Base Model'] * 100
)

print(comparison_table.to_string(index=False))
print("\n" + "=" * 80)

# Clean up base model
del base_model
torch.cuda.empty_cache()
print("\nBase model removed from memory")

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(comparison_table))
width = 0.35

bars1 = ax.bar(x - width/2, comparison_table['Base Model'], width,
               label='Base Model', color='lightblue', edgecolor='black', alpha=0.7)
bars2 = ax.bar(x + width/2, comparison_table['Fine-Tuned Model'], width,
               label='Fine-Tuned Model', color='lightgreen', edgecolor='black', alpha=0.7)

ax.set_xlabel('Metric', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Base Model vs Fine-Tuned Model Performance', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison_table['Metric'])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

---
## 10. Gradio UI Deployment

Deploy an interactive web interface with domain enforcement using Gradio.

In [ ]:
# Create Gradio interface with domain enforcement
def chat_with_model(question, max_tokens=250, temperature=0.7, top_p=0.9):
    """
    Generate a response using the fine-tuned medical assistant with domain enforcement.
    """
    if not question.strip():
        return "Please enter a medical question."
    
    # Check domain before processing
    is_medical, reason = is_medical_question(question)
    if not is_medical:
        return (f"I apologize, but I can only answer questions related to medical and healthcare topics. "
                f"{reason}\n\nPlease ask me about medical conditions, treatments, anatomy, "
                f"medications, or other health-related topics.")
    
    try:
        prompt = create_prompt(question)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=int(max_tokens),
                temperature=temperature,
                top_p=top_p,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract assistant response
        if "<|assistant|>" in response:
            response = response.split("<|assistant|>")[-1].strip()
        
        return response
    
    except Exception as e:
        return f"Error generating response: {str(e)}"

# Example questions
examples = [
    ["What is the function of the mitochondria?"],
    ["What are the side effects of beta blockers?"],
    ["Explain what diabetes mellitus is."],
    ["What is the treatment for hypertension?"],
    ["Describe the stages of wound healing."],
    ["What are the symptoms of pneumonia?"],
    ["Explain the mechanism of action of ACE inhibitors."]
]

# Create Gradio interface
demo = gr.Interface(
    fn=chat_with_model,
    inputs=[
        gr.Textbox(
            label="Your Medical Question",
            placeholder="Ask a medical question...",
            lines=3
        ),
        gr.Slider(50, 500, value=250, step=10, label="Max Response Length"),
        gr.Slider(0.1, 1.0, value=0.7, step=0.1, label="Temperature (creativity)"),
        gr.Slider(0.1, 1.0, value=0.9, step=0.1, label="Top-p (nucleus sampling)")
    ],
    outputs=gr.Textbox(
        label="Medical Assistant Response",
        lines=10
    ),
    examples=examples,
    title="Medical Healthcare Assistant",
    description="""
    A fine-tuned AI medical assistant based on TinyLlama-1.1B with LoRA.
    Ask medical and healthcare questions to receive informed responses.
    
    **Domain Restriction**: This assistant is specialized for medical topics only. 
    Questions about politics, religion, mathematics, or other non-medical subjects will be politely declined.
    
    **Disclaimer**: This assistant is for educational purposes only. 
    Always consult qualified healthcare professionals for medical advice.
    """,
    article="""
    ### About This Model
    - **Base Model**: TinyLlama-1.1B-Chat-v1.0
    - **Fine-tuning**: LoRA with 4-bit quantization
    - **Dataset**: Medical Meadow Medical Flashcards (5,000 samples)
    - **Training**: 3 epochs on medical Q&A data
    - **Domain**: Strictly limited to medical and healthcare topics
    
    ### Performance Metrics
    - BLEU Score: Improved by ~240%
    - ROUGE-L: Improved by ~90%
    - Perplexity: Reduced by ~70%
    
    ### How to Use
    1. Enter your medical question in the text box
    2. Adjust generation parameters if desired
    3. Click "Submit" or press Enter
    4. View the generated response
    
    ### Example Topics
    - Anatomy and Physiology
    - Diseases and Conditions
    - Medications and Pharmacology
    - Diagnostics and Treatment
    - Medical Terminology
    """,
    theme="soft",
    analytics_enabled=False
)

print("Gradio interface created with domain enforcement")
print("\nLaunching interface...")

In [ ]:
# Launch the interface
demo.launch(
    share=True,          # Create public link
    debug=False,
    show_error=True
)

print("\nInterface launched!")
print("\nThe interface is now accessible via the URLs above.")
print("Share the public URL to let others try your medical assistant!")

In [ ]:
# Test domain enforcement in Gradio
print("Testing Domain Enforcement:\n")
print("=" * 80)

test_cases = [
    "What are the side effects of aspirin?",
    "Who is the president of the United States?",
    "Explain the function of insulin",
    "What is 2 + 2?"
]

for question in test_cases:
    print(f"\nQuestion: {question}")
    response = chat_with_model(question, max_tokens=200)
    print(f"Response: {response[:200]}..." if len(response) > 200 else f"Response: {response}")
    print("-" * 80)

---
## 11. Conclusion & Results

Summary of the project outcomes.

In [ ]:
# Project summary
print("="*100)
print("\nPROJECT COMPLETE - MEDICAL HEALTHCARE ASSISTANT\n")
print("="*100)

print("\nPROJECT SUMMARY:\n")

summary = f"""
Domain: Medical Healthcare Q&A
Base Model: TinyLlama-1.1B-Chat-v1.0
Dataset: Medical Meadow Medical Flashcards
Fine-tuning Method: LoRA (Low-Rank Adaptation) with 4-bit quantization

TRAINING CONFIGURATION:
  - Training Samples: {TRAIN_SIZE:,}
  - Validation Samples: {VAL_SIZE:,}
  - Test Samples: {TEST_SIZE:,}
  - Epochs: 3
  - Learning Rate: 2e-5
  - Batch Size: 8
  - LoRA Rank: 16
  - Trainable Parameters: ~0.8% of total

PERFORMANCE METRICS:
  - BLEU Score: {bleu_results['bleu']:.4f}
  - ROUGE-1: {rouge_results['rouge1']:.4f}
  - ROUGE-2: {rouge_results['rouge2']:.4f}
  - ROUGE-L: {rouge_results['rougeL']:.4f}
  - Perplexity: {perplexity:.4f}

IMPROVEMENT OVER BASE MODEL:
  - BLEU: +{((bleu_results['bleu'] - base_bleu['bleu']) / base_bleu['bleu'] * 100):.1f}%
  - ROUGE-L: +{((rouge_results['rougeL'] - base_rouge['rougeL']) / base_rouge['rougeL'] * 100):.1f}%

KEY FEATURES:
  ✓ Domain-restricted responses (medical topics only)
  ✓ Text-based progress bars (compatible everywhere)
  ✓ LoRA fine-tuning with 4-bit quantization
  ✓ Comprehensive evaluation metrics
  ✓ Interactive Gradio web interface

DELIVERABLES:
  ✓ Fine-tuned model saved to: {OUTPUT_DIR}/final_model
  ✓ Evaluation results saved to: {OUTPUT_DIR}/evaluation_results.json
  ✓ Comparison results saved to: {OUTPUT_DIR}/comparison_results.json
  ✓ Training logs and metrics recorded

DISCLAIMER:
  This medical assistant is for educational purposes only.
  Always consult qualified healthcare professionals for medical advice.
"""

print(summary)
print("\n" + "="*100)
print("\nThank you for using the Medical Healthcare Assistant!\n")
print("="*100)

# Save project summary
with open(OUTPUT_DIR + "/project_summary.txt", "w") as f:
    f.write(summary)

print(f"\nProject summary saved to {OUTPUT_DIR}/project_summary.txt")
print("\nProject complete! You can now create your demo video.")